In [1]:
import os
import shutil
import random
from PIL import Image
from torchvision import transforms
import numpy as np

**Dataset Preprocessing**

In [2]:
sources = ["open_data_set\photos_all_faces",
           "open_data_set\portraits",
           "open_data_set\\trio_gp",
           "open_data_set\\trio_cam"]

destination = "combined_dataset"
os.makedirs(destination, exist_ok=True)

for source in sources:
    for filename in os.listdir(source):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            src_path = os.path.join(source, filename)
            dst_path = os.path.join(destination, filename)
            shutil.copy(src_path, dst_path)

print("all images combined successfully")


all images combined successfully


In [3]:


source = "combined_dataset"
destination = "prepared"

os.makedirs(destination, exist_ok=True)

for filename in os.listdir(source):
    if filename.lower().endswith((".jpg", ".png", "jpeg")):
        identity = filename[0].upper()
        identity_folder = os.path.join(destination, identity)
        os.makedirs(identity_folder, exist_ok=True)
        src_path = os.path.join(source, filename)
        dst_path = os.path.join(identity_folder, filename)
        shutil.move(src_path, dst_path)

print("Data split by identity")

Data split by identity


In [4]:
source = "prepared"
base_split = "split" 

#splitting ids into training, validation, and testing
train_ids = ["A", "B", "C", "D", "E", "F", "G", "H"] #70% Training
val_ids = ["I"] #10% validation
test_ids = ["J", "K"] #20% testing


for split, ids in [("train", train_ids), ("validation", val_ids), ("test", test_ids)]:
    for identity in ids: 
        src = os.path.join(source, identity)
        dst = os.path.join(base_split, split, identity)
        os.makedirs(dst, exist_ok=True)
        for file in os.listdir(src):
            shutil.copy(os.path.join(src, file),
                        os.path.join(dst, file))
            
print("identity disjoint split completed")



identity disjoint split completed



## Data Transformation

In [5]:
from torch.utils.data import Dataset 
import torchvision.transforms as transforms

class DroneFaceDataset(Dataset):
    def __init__(self, root_dir): 
        self.root_dir = root_dir
        self.classes = sorted(os.listdir(root_dir))
        self.image_paths = []
        self.labels = []

        for label, class_name in  enumerate(self.classes):
            class_Folder = os.path.join(root_dir, class_name)
            for img in os.listdir(class_Folder):
                self.image_paths.append(os.path.join(class_Folder, img))
                self.labels.append(label)
                
        #applying data transformations 

        self.transform = transforms.Compose([transforms.Resize((112, 112)), 
                                             transforms.RandomHorizontalFlip(),
                                             transforms.RandomRotation(20),
                                             transforms.GaussianBlur(kernel_size=3),
                                             transforms.RandomResizedCrop(112, scale=(0.8,1.0)),
                                             transforms.ColorJitter(brightness=0.2, contrast=0.2),
                                             transforms.ToTensor(),
                                             transforms.Normalize(mean=[0.5, 0.5, 0.5],
                                                                  std = [0.5, 0.5, 0.5]),
                                            ])
        
    def __len__(self):
            return len(self.image_paths)
        
    def __getitem__(self, idx):
            img = Image.open(self.image_paths[idx]).convert("RGB")
            label = self.labels[idx]
            img = self.transform(img)

            return img, label
        


## Model Development

In [6]:
from deepface import DeepFace
import tf_keras
import cv2

# Test on one image from your dataset
img_path = "split\\train\\A"
first_img = os.path.join(img_path, os.listdir(img_path)[0])

embedding = DeepFace.represent(
    img_path    = first_img,
    model_name  = "ArcFace",   # best model for face recognition
    enforce_detection = False
)

print("✅ DeepFace working")
print(f"Embedding size: {len(embedding[0]['embedding'])}")


26-03-22 20:50:09 - arcface_weights.h5 will be downloaded to C:\Users\Shamm\.deepface/weights\arcface_weights.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: C:\Users\Shamm\.deepface\weights\arcface_weights.h5
100%|██████████| 137M/137M [00:25<00:00, 5.38MB/s] 


✅ DeepFace working
Embedding size: 512


In [7]:
import torch
import torch.nn as nn 
import torchvision.models as models 
import torch.nn.functional as F

class DroneFaceRecognitionModel(nn.Module):
    def __init__(self, num_classes): 
        super().__init__()
        self.backbone = models.resnet18(weights = models.ResNet18_Weights.IMAGENET1K_V1)
    
        self.embedding = nn.Linear(512, 512) #embedding layer
        self.backbone.fc = nn.Identity()
        

    def get_Embedding(self,x):
        feat=self.backbone(x)
        emb = self.embedding(feat)
        return F.normalize(emb, p=2, dim=1)
    
    def forward(self, x):
        return self.get_Embedding(x)

In [8]:
import torch.nn.functional as F 
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=64.0, m=0.5): 
        # in_features = embedding size
        # num_classes=number of classes in our dataset
        # s = scaling factor
        # m = angular margin
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight) #one learnable vector per person in embedding space
    def forward(self, features, labels):
        cosine_theta = f.linear(f.normalize(features), f.normalize(self.weight)) #measures similarity
        theta = torch.acos(cosine_theta.clamp(-1 + 1e-7,  1 - 1e-7)) #converting to an angle
        one_hot_encode = torch.zeros_like(cosine_theta) #mask creation for image classification
        one_hot_encode.scatter_(1, labels.view(-1,1), 1)
        output = torch.where(one_hot_encode.bool(), torch.cos(theta 
                                                            + self.m), cosine_theta)  #improves model training
        
        return output * self.s #scaled by 64 for effective training
    




**Data Loading**

In [9]:
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

#dataset loading
train_set = DroneFaceDataset("split\\train")
validation_set = DroneFaceDataset("split\\validation")
test_set = DroneFaceDataset("split\\test")

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

num_classes = len(train_set.classes)


In [10]:
model = DroneFaceRecognitionModel(num_classes).to(device)
classifier = nn.Linear(512, num_classes).to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False #freezing backbone

#unfreezing layer 2, 3, 4, and final layer
for paran in model.backbone.layer2.parameters():
    param.requires_grad = True
for paran in model.backbone.layer3.parameters():
    param.requires_grad = True
for param in model.backbone.layer4.parameters():
    param.requires_grad = True
for param in model.backbone.fc.parameters():
    param.requires_grad = True


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([{'params': filter(lambda p: p.requires_grad, model.parameters()), 'lr':1e-4},
                              {'params': classifier.parameters(), 'lr': 1e-3}])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


In [11]:
#Accuracy Evaluation
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0 

    with torch.no_grad():
        for images, labels in loader:
            images=images.to(device)
            labels=labels.to(device)
            outputs=model(images)
            _,predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted==labels).sum().item()
    return 100 * correct / total

In [ ]:



best_valacc = 0.0

for epoch in range(30):
    model.train()
    classifier.train()
    total_loss = 0

    for images, labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()
        embeddings=model(images)
        logits = classifier(embeddings)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss +=loss.item()
    scheduler.step()
    #building gallery
    model.eval()
    classifier.eval()
    embeddings_gallery, labels_gallery = [], []
    with torch.no_grad():
        for images, labels in train_loader:
            embeds = model(images.to(device))
            embeddings_gallery.append(embeds.cpu())
            labels_gallery.append(labels)
    embeddings_gallery = torch.cat(embeddings_gallery)
    labels_gallery = torch.cat(labels_gallery)
#training accuracy using cosine similarity
    top1_train = total_train = 0
    # with torch.no_grad():
    #     for images, labels in train_loader:
    #         embeddings = F.normalize(model(images.to(device)), p=2, dim=1).cpu()
    #         for i in range(len(embeddings)):
    #             similarities = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings_gallery)
    #             predictions = labels_gallery[torch.argmax(similarities)].item()
    #             if predictions == labels[i].item():
    #                 top1_train+=1
    #             total_train +=1
    #     train_accuracy = 100 * top1_train/ total_train


    #validation accuracy using cosine similarity (Different people)
    top1_val = total_val = 0 
    with torch.no_grad():
        for images, labels in validation_loader:
            embeddings = F.normalize(model(images.to(device)), p=2, dim=1).cpu()
            for i in range(len(embeddings)):
                similarities = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings_gallery)
                predictions = labels_gallery[torch.argmax(similarities)].item()
                if predictions == labels[i].item():
                    top1_val+=1
                total_val +=1
        validation_accuracy = 100 * top1_val/ total_val
                               
    
    #to save best model 
    if validation_accuracy > best_valacc:
        best_valacc = validation_accuracy
        torch.save({'model': model.state_dict(), 'classifier': classifier.state_dict()}, "best_model.pth")
        print("best model saved")
    print(f"Epoch: {epoch+1}    Loss: {total_loss:.4f}    Validation Accuracy: {validation_accuracy:.2f}%")
    print(f" Best Validation Accuracy: {best_valacc:.2f}%")

   
    
print(f"Best validation accuracy: {best_valacc:.2f}%")    

Epoch: 1    Loss: 63.9713    Validation Accuracy: 0.00%
 Best Validation Accuracy: 0.00%
best model saved
Epoch: 2    Loss: 46.4083    Validation Accuracy: 0.76%
 Best Validation Accuracy: 0.76%


In [ ]:


#load 
checkpoint = torch.load("best_model.pth")
model.load_state_dict(checkpoint['model'])
model.eval()




KeyError: 'model'

**Embeddings Generation and Storage**

In [ ]:
#function for generating embeddings 
def embedding(image, model, device):
    model.eval() 
    image = image.unsqueeze(0).to(device)
    with torch.no_grad():
        feature = model(image)
    embedding= f.normalize(feature, p=2, dim=1)
    return embedding.cpu()

In [ ]:
#embeddings for training data 
embeddings_set = []
labels_set = []

with torch.no_grad():
    for images, labels in train_loader:
        images  = images.to(device)
        features = model(images)
        embeddings = F.normalize(features, p=2, dim=1)
        embeddings_set.append(embeddings.cpu())
        labels_set.append(labels)
embeddings_set = torch.cat(embeddings_set, dim=0)
labels_set = torch.cat(labels_set, dim=0)


In [ ]:
torch.save({"embeddings": embeddings_set, "labels": labels_set}, "face_database.pth")

**Face Recognition using Cosine_Similarity**

In [ ]:
from torch.nn.functional import cosine_similarity
#comparing test images to images in database 

def recognize(query_embedding, embeddings_set):
    similarities = cosine_similarity(query_embedding, embeddings_set)
    best_match = torch.argmax(similarities)
    score = similarities[best_match]
    return best_match, score


In [ ]:
top1_correct = 0
top5_correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        features = model(images)
        query_embedding = F.normalize(features, p=2, dim=1).cpu()

        for i in range(len(query_embedding)):
            similarities = F.cosine_similarity(query_embedding[i].unsqueeze(0), embeddings_gallery)
            top5_ids= torch.topk(similarities, k=min(5, len(similarities))).indices
            predicted = labels_gallery[top5_ids[0]].item()
            top5_predicted = [labels_gallery[j].item() for j in top5_ids]
            true_label = labels[i].item()

            if predicted == true_label:
                top1_correct +=1
            if true_label in top5_predicted:
                top5_correct +=1
            total +=1
print(f"Rank 1 Accuracy: {100*top1_correct/total:.2f}%")
print(f"Rank 5 Accuracy: {100*top5_correct/total:.2f}%"
      )

Rank 1 Accuracy: 5.73%
Rank 5 Accuracy: 22.52%
